# RAG chatbot demonstration
This notebook demonstates the capability of a chat bot with RAG

In [1]:
import os
import re

import certifi
import duckdb
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from electricity_maps.config import get_settings

e:\IntelliJ\intelliWorkspace\ElectricityMaps\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Setup Environment and Helper Functions
os.environ["AWS_CA_BUNDLE"] = certifi.where()
os.environ["SSL_CERT_FILE"] = certifi.where()

get_settings.cache_clear()
settings = get_settings()
DATA_DIR = settings.data_dir


def get_sql_df(sql_query):
    # Use duckdb to query Delta Lake
    def replace_table(match):
        keyword = match.group(1)
        schema = match.group(2)
        table = match.group(3)
        return f"{keyword} delta_scan('{DATA_DIR}/{schema}/{table}')"

    parsed_query = re.sub(
        r'(?i)(FROM|JOIN)\s+([a-zA-Z0-9_]+)\.([a-zA-Z0-9_]+)',
        replace_table,
        sql_query
    )

    con = duckdb.connect()
    # If AWS S3 is used, you would load aws extension here
    con.execute("INSTALL httpfs; LOAD httpfs; INSTALL aws; LOAD aws;")
    con.execute(f"CREATE OR REPLACE SECRET aws_s3 (TYPE S3, KEY_ID '{settings.aws_access_key_id}', SECRET '{settings.aws_secret_access_key}', REGION '{settings.aws_region}');")
    return con.execute(parsed_query).df()

def print_sql_df(sql_query):
    df = get_sql_df(sql_query)
    display(df)
    return df

gold_table = get_sql_df("SELECT count(*) as count FROM gold.daily_electricity_mix")
display(gold_table)


,count
0,3


In [3]:
# 2. Data schema and LLM Initialization
gold_schemas = {
    "daily_electricity_mix": {
        "description": "Daily electricity mix data with percentages of different energy sources",
        "columns": ["process_ts", "zone", "zone_name", "date", "nuclear_pct", "biomass_pct", "wind_pct", "solar_pct", "hydro_pct", "gas_pct", "oil_pct", "coal_pct", "geothermal_pct", "unknown_pct", "total_production_mwh", "fossil_free_avg_pct", "renewable_avg_pct", "hours_covered", "year", "month", "day"]
    },
    "daily_imports": {
        "description": "Daily electricity imports data",
        "columns": ["process_ts", "zone", "zone_name", "source_zone", "source_zone_name", "date", "import_mwh", "hours_covered", "year", "month", "day"]
    },
    "daily_exports": {
        "description": "Daily electricity exports data",
        "columns": ["process_ts", "zone", "zone_name", "destination_zone", "destination_zone_name", "date", "export_mwh", "hours_covered", "year", "month", "day"]
    }
}

print(f"Model: {settings.llm_model}")
llm = ChatGoogleGenerativeAI(api_key=settings.llm_api_key, model=settings.llm_model)

schema_context = "\n".join([f"Table: gold.{table}\nDescription: {info['description']}\nColumns: {', '.join(info['columns'])}\n" for table, info in gold_schemas.items()])


Model: gemini-3.1-flash-lite-preview


In [4]:
# 3. Process docs folder and create embeddings in ChromaDB
docs_path = "../docs/Electricity_Maps_doc.pdf"
loader = PyPDFLoader(docs_path)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(documents)

# Switching to Google's text-embedding-004 model
embeddings = GoogleGenerativeAIEmbeddings(api_key=settings.llm_api_key, model="models/text-embedding-004")
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings, persist_directory="./chroma_db")
retriever = vectorstore.as_retriever()


C:\Users\user\AppData\Local\Temp\ipykernel_2496\1965984186.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8832.28it/s]


In [5]:
# 4. Agent Setup: Router, Text-to-SQL, and RAG

# --- Router ---
router_template = """You are a query routing agent.
Given the following user query, decide if it should be routed to a structured database (SQL) or an unstructured document database (RAG).
Route to 'SQL' if the query asks for metrics, trends, percentages, data aggregation, imports/exports, or numerical electricity data.
Route to 'RAG' if the query asks for definitions, documentation, company information, methodology, FAQs, or explanations.

Return ONLY the string 'SQL' or 'RAG'.
Query: {query}
Route:"""
router_prompt = PromptTemplate.from_template(router_template)
router_chain = router_prompt | llm | StrOutputParser()

# --- Text-to-SQL Chain ---
sql_template = """You are a data analyst. Convert the natural language question into a valid SQL query for DuckDB.
Only use the following schemas and tables.
Schema:
{schema}

IMPORTANT:
1. Always output ONLY the raw SQL query, with no markdown formatting or explanations. Do not include ```sql blocks.
2. If there's a reference to a latest process_ts, you can optionally filter by it if relevant.

Question: {question}
SQL Query:"""
sql_prompt = PromptTemplate.from_template(sql_template)
sql_chain = sql_prompt | llm | StrOutputParser()

# --- Document RAG Chain ---
rag_template = """Answer the question based only on the following context:
{context}

Question: {question}
Answer:"""
rag_prompt = ChatPromptTemplate.from_template(rag_template)
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)


In [6]:
# 5. Main Chatbot Interface
def ask_chatbot(question: str):
    print(f"User: {question}")

    # 1. Route the query
    route = router_chain.invoke({"query": question}).strip().upper()
    print(f"[Router decision: {route}]")

    # 2. Execute path
    if route == "SQL":
        # Generate SQL
        sql_query = sql_chain.invoke({"schema": schema_context, "question": question}).strip()
        # Clean up markdown formatting if the LLM adds it anyway
        sql_query = sql_query.replace('```sql', '').replace('```', '').strip()
        print(f"Generated SQL: {sql_query}")

        try:
            print("Executing query against Gold Layer...")
            print_sql_df(sql_query)
        except Exception as e:
            print(f"Error executing SQL: {e}")

    else:
        # Document RAG
        print("Querying Document Knowledge Base...")
        response = rag_chain.invoke(question)
        print(f"Chatbot: {response}")
    print("-" * 50)


In [7]:
# 6. Test the Chatbot
# Test 1: Ask for structured data (SQL Route)
ask_chatbot("What is the average renewable percentage for France?")



User: What is the average renewable percentage for France?
[Router decision: SQL]
Generated SQL: SELECT AVG(renewable_avg_pct) FROM gold.daily_electricity_mix WHERE zone_name = 'France'
Executing query against Gold Layer...


,avg(renewable_avg_pct)
0,25.248749


--------------------------------------------------


In [8]:
# Test 2: Ask for unstructured documentation (RAG Route)
ask_chatbot("What is Electricity Maps methodology?")

User: What is Electricity Maps methodology?
[Router decision: RAG]
Querying Document Knowledge Base...
Chatbot: Based on the provided context, it is not mentioned what the methodology is. The text only notes that there are slight delays because data is reported from the last reporting interval and there are technical delays in delivering that data.
--------------------------------------------------


In [9]:
# Test 3: Complex SQL query
ask_chatbot("Show me the top 3 destinations for electricity exported from France")

User: Show me the top 3 destinations for electricity exported from France
[Router decision: SQL]
Generated SQL: SELECT destination_zone_name, SUM(export_mwh) as total_exported FROM gold.daily_exports WHERE zone_name = 'France' GROUP BY destination_zone_name ORDER BY total_exported DESC LIMIT 3
Executing query against Gold Layer...


,destination_zone_name,total_exported
0,Great Britain,118289.0
1,Italy North,67647.0
2,LU,7236.0


--------------------------------------------------
